## Scipy and Optimization

In the last notebook we took a tour of the `numpy` and `scipy` stack as it relates to math and statistics. This notebook is going to focus on another subset of `scipy` functionality that is absolutely critical to data analytics. Nearly every statistical algorithm aside from Ordinary Least Squares (OLS) regression cannot be solved algebraically. Because they cannot be solved algebraically, they must be solved **numerically**.

### Numeric Optimization

Let's start with an example of optimization. Let's say that you know the demand function for tickets to watch a local sports franchise. You can write the inverse demand function as

$$ P = 300 - \frac{1}{2}Q $$

and the total cost function as

$$ TC = 4000 + 45Q $$

In order to choose the right number of tickets to sell, you need to calculate the quantity of tickets that will maximize profits. We can calculate total revenue as $ TR = P \times Q $, and we can calculate profit as $ \Pi = TR - TC $. This means that our profit function is

$$ \Pi = 300Q - \frac{1}{2}Q^2 - 4000 - 45Q $$

In order to find the $Q$ associated with the highest achievable level of profit, we can use calculus to find the point at which the rate of change in the profit function is zero ($\frac{\partial\Pi}{\partial Q}=0$).

$$ \frac{\partial\Pi}{\partial Q} = 300 - Q - 45 = 0 \implies Q = 255$$

So we can **algebraically** solve this particular problem. This isn't always the case. Using `scipy`, we can solve this same problem, as well as many algebraically intractable problems that might be more interesting to us.


<img src="https://github.com/dustywhite7/Econ8320/raw/master/SlidesCode/paraboloid.png" width="200" height="200" />


In any optimization problem, we need to find a way to get ourselves to the minimum, and to know when we get there. When we look at the above image, we are able to visually trace the functional shape (looks like a rainbow ice cream cone to me...) and locate the bottom of the function. What we want to do is utilize an algorithm to "trace" our way from an arbitrary starting point within a function to the optimal point in that function.

In three or fewer dimensions, this is easy. Regressions and statistical models often live in worlds with 100's or 1000's (even millions sometimes) of dimensions. We can't visualize our way to the bottom of those functions!

The class of algorithm that is used to solve these problems is called **gradient descent**.

<img src="https://github.com/dustywhite7/Econ8320/raw/master/SlidesCode/gradDesc.png" width="400" />

**Gradient descent** is an algorithm that explores the shape of the function, and determines which direction is most likely to lead to the optimal point. Let's focus on minimization. We want to find our way to the *bottom* of a function, and we can use gradient descent to try to get there. Given any starting point, our goal is to check the direction in which we can move downward most quickly, and start moving in that direction. At some distance from our starting point, we will stop and re-evaluate the direction in which we would like to travel. Are we still heading downhill in the steepest direction? If we aren't, then we need to update our behavior.

Our gradient descent algorithm will keep "looking around" and moving down until it reaches a point at which it can no longer move "down" in any meaningful way. That is the stopping point, and is treated as the optimum.


With an intuitive understanding of how optimization will happen computationally, it's time to learn a bit more about the math and the code that will help us to achieve computational optimization.

Consider a function, $f$, with two variables $x$ and $y$. Because there are two input variables in the function, it has two partial derivatives:

$$ \frac{\partial f}{\partial x} \text{ and } \frac{\partial f}{\partial y} $$

Each partial derivative tells us how $f$ changes as we move in a particular dimension **all else equal**. The **gradient**, then, is the vector of all partial derivatives of a given function at any point along the function:

$$ \nabla f = \left[ \begin{matrix} \frac{\partial f}{\partial x} \\ \\ \frac{\partial f}{\partial y} \end{matrix} \right]  $$

We can use the gradient to determine the linear approximation of a function at any given point. Think about the gradient as the mathematical representation of the slope and direction of a hill you are hiking on. If we know the gradient, we know which way is down. As we continue to calculate gradients while walking, we can continue to ensure that we will walk downhill until we reach the bottom of the hill.




The steps of our gradient descent function will be the following:

- Evaluate the gradient of the function
- Find the direction of steepest descent
- Determine how far to move in that direction
- Move to new point
- Reevaluate the gradient
- Stop moving when gradient is within a margin of error from 0

Let's try to implement gradient descent by solving our old profit maximization problem computationally. The very first thing that we need to do is write a Python function that will represent our mathematical function.

In [ ]:
def profit(q):
    p = 300-0.5*q
    tr = p*q
    tc = 4000 + 45*q
    return tr - tc

This function will allow us to calculate profit at any output level based on our assumed total costs and demand curve. With this function, we can quickly calculate the gradient (in this case, just a simple derivative because our function is univariate) by calculating profit at two nearby points, and dividing by the horizontal distance between those points:

In [ ]:
# Gradient at q=200

(profit(201) - profit(199))/2

    55.0



Thus, a one unit increase in output at $Q=200$ results in a $55 increase in profits. This is cool, but it isn't enough for us to find the point of maximum profit (the optimal point). For that, we will need to calculate LOTS of gradients in order to move along the function until we cannot increase profits any further.

Fortunately for us, `scipy` comes with optimization tools that will do all of the heavy lifting of the "search" for the optimal point. All that we need to do is frame our question algorithmically, and let `scipy` do the rest:

In [ ]:
import numpy as np
from scipy.optimize import minimize

We start by importing the `minimize` function from `scipy.optimize`. Hang on! Weren't we working on a MAXIMIZATION problem?? What are we doing here?

Maximization and minimization are the **same thing**. To maximize a function, you can multiply that function by `-1` and then calculate the minimum of the new "upside-down" function. It is functionally equivalent. So, in computational optimization, we always minimize.

### Prepping for optimization

As we prepare to optimize, there are two common problems with our function that we may need to resolve:

1) When using `minimize` we can only pass an array of inputs, so we have to be careful to write our function accordingly
2) Our problem is concave, and so has a maximum
	- We need to restate it as a minimization problem

Problem 1 does not apply, since our function in univariate. In order to make our problem a minimization problem, we can flip our profit maximization function. We will simply return -1 * profit:

In [ ]:
def profit(q):
    p = 300-0.5*q
    tr = p*q
    tc = 4000 + 45*q
    pi =  tr - tc # didn't name it profit because that is our function's name! Don't want to clutter our name space!
    return -1*pi

### Making the call to `minimize`

Now that our function is ready, it is time to minimize! The `minimize` function takes two arguments:
1. Our function that we want to optimize
2. A starting guess (as a vector)

Let's give it a try.

In [ ]:
res = minimize(profit, [0]) # provide function and starting inputs
res

          fun: -28512.499999980355
     hess_inv: array([[1.00000175]])
          jac: array([0.])
      message: 'Optimization terminated successfully.'
         nfev: 21
          nit: 3
         njev: 7
       status: 0
      success: True
            x: array([255.00019821])



That's it! No calculus, no searching, no checking gradients manually. `minimize` simply takes our function and our starting guess and brings us back the optimal choice. We get lots of information stored in the attributes of the `res` object:

- `fun` provides the value of the function (this is -1 times the profit level at the optimal output in our example)
- `hess_inv` and `jac` are measures of gradient and are used to determine how far to go and in which direction
- `message` should be self-explanatory
- `nfev` is the number of times the function (in this case `profit`) was evaluated during the search
- `nit` is the number of iterations it took to find the optimum
- `njev` is the number of times the Jacobian was estimated
- `status` is a code associated with the `message` and `success` atrributes
- `success` tells you whether or not an optimum was found (sometimes it cannot be easily found!)
- `x` probably the most important attribute. This tells us the optimal input value (in our case $Q$), or set of values depending on our function. In a regression context, this could represent the fitted coefficients!

Going forward, you will realize (especially because so many of them print the output as they optimize) just how many libraries in Python use this minimize function behind the scenes. It is used in `statsmodels`, `sklearn`, and many other high-profile libraries! Now that you know where it really happens (in `scipy`!), you'll be better able to troubleshoot the problems that will inevitably arise as you use statistical models.

**Solve-it!**

In this lesson we learned about optimization using SciPy. For the assignment this week, I would like you to build off of your `RegressionModel` class. You will add a Logistic Regression (Logit) method to your class, so that when the `regression_type` parameter is `logit`, Logistic Regression Results are returned.

Your job is to create the following functionality within your class object:
- a method (call it `logistic_regression`) that estimates the results of logistic regression using your `x` and `y` data frames, and using a likelihood function and gradient descent (DO NOT USE PREBUILT REGRESSION FUNCTIONS).
    - You need to write a function to calculate the Log-likelihood of your model
    - You need to implement gradient descent to find the optimal values of beta
    - You need to use your beta estimates, the model variance, and calculate the standard errors of the coefficients, as well as Z statistics and p-values
    - the results should be stored in a dictionary named `results`, where each variable name (including the intercept if `create_intercept` is `True`) is the key, and the value is another dictionary, with keys for `coefficient`, `standard_error`, `z_stat`, and `p_value`. The coefficient should be the log odds-ratio (which takes the place of the coefficients in OLS)
- a method called `fit_model` that uses the `self.regression_type` attribute to determine whether or not to run an OLS or Logistic Regression using the data provided. This method should call the correct regression method.
- an updated method (call it `summary`) that presents your regression results in a table
    - Columns should be: Variable name, Log odds-ratio value, standard error, z-statistic, and p-value, in that order.
    - Your summary table should have different column titles for OLS and Logistic Regressions! (think if statement...)

You only need to define the class. My code will create an instance of your class (be sure all the names match these instructions, and those from last week!), and provide data to run a regression. I will provide the same data to you, so that you can experiment and make sure that your code is functioning properly.

NOTE: I have created a [primer on Logistic regression](https://github.com/dustywhite7/Econ8320/blob/master/SlidesPDF/9-2%20-%20Logit%20Primer.pdf) to go with this assignment. See the Github slidesPDF folder

Put the code that you would like graded in the cell labeled `#si-exercise`. I recommend copying your code from the last assignment (in chapter 9 about linear regression and `numpy`), and continuing from there.



In [124]:
import numpy as np
import pandas as pd
from scipy.stats import t
from scipy.stats import norm
from scipy.optimize import minimize

data = pd.read_csv("https://github.com/dustywhite7/Econ8320/raw/refs/heads/master/AssignmentData/assignment8Data.csv")
x = data.loc[:100, ['sex','age','educ']]
x['intercept']=1
y = data.loc[:100, 'white']

In [ ]:
#create a separate logistic transformation function: exp(x'β)/[1+exp(x'β)] = Λ(x'β)
#this coerces the output to remain between 0 and 1
#per Dusty, use x (shape 101,4) instead of x' (shape 4,101) because of Dot product shape mismatch: (4, 101) vs (4, 1)

In [186]:
#si-exercise
import numpy as np
import pandas as pd
from scipy.stats import t
from scipy.stats import norm
from scipy.optimize import minimize

"""
type(x) = data frame [['sex','age','educ']]
type(y) = data frame ['white']
type(create_intercept) = boolean
type(regression_type) = str (either "ols" or "logit")
"""

class RegressionModel(object):

  def __init__(self, x, y, create_intercept=True, regression_type='logit'):
    self.x = x
    self.y = y
    self.create_intercept = create_intercept
    self.regression_type = regression_type
    self.x['intercept'] = 1
    self.fit_model()

  #a method that calls ols_regression() or logistic_regression() based on self.regression_type
  def fit_model(self):
    if self.regression_type == 'logit':
      return self.logistic_regression()
    elif self.regression_type == 'ols':
      return self.ols_regression()
    else:
      raise RuntimeError("regression_type must be a string with input 'ols' or 'logit'.")

  # def ols_regression(self):

  #     #break up components of beta calculation: β = (x'x)^-1 x'y
  #     xtrans_x = self.x.T @ self.x #5x5
  #     inv_xtrans_x = np.linalg.inv(xtrans_x) #5x5
  #     xtrans_y = self.x.T @ self.y #5x1

  #     #calculate beta
  #     self.beta = inv_xtrans_x @ xtrans_y #5x1

  #     #break up components of variance estimate [(y-xβ)'(y-xβ)]/(n-k)
  #     xbeta = self.x @ self.beta #13712x1
  #     y_less_xbeta = pd.DataFrame(self.y - xbeta) #13712x1
  #     transy_less_xbeta = y_less_xbeta.T #1x13712
  #     df = self.x.shape[0] - self.x.shape[1] #int
  #     shat_numer = transy_less_xbeta @ y_less_xbeta #1x1

  #     #calculate unbiased variance estimate, shat^2 (a scalar)
  #     shat_sqrd = shat_numer / df #1x1

  #     #calculate covariance (shat_sqrd (x'x)^-1)
  #     covar = shat_sqrd.values * inv_xtrans_x #the main diagonal is the variance of each β coefficient. SE is the sqr root of the variance. (5x5)

  #     #we only need the diagonal (variance) from the covariance matrix
  #     #calculate the variance
  #     variance = np.diag(covar) #1x5

  #     #calculate the standard error
  #     self.stdErr = np.sqrt(variance) #1x5

  #     #calculate the t-stat
  #     self.tstat = self.beta / self.stdErr #1x5

  #     #calculate the p-value
  #     self.pval = t.sf(self.tstat, df) #1x5

  #     #create a nested dictionary to store results for each variable
  #     self.results = {
  #         "sex": {"coefficient":self.beta[0],
  #                 "standard_error":self.stdErr[0],
  #                 "t_stat":self.tstat[0],
  #                 "p_value":self.pval[0]
  #                 },
  #         "age": {"coefficient":self.beta[1],
  #                 "standard_error":self.stdErr[1],
  #                 "t_stat":self.tstat[1],
  #                 "p_value":self.pval[1]
  #                 },
  #         "educ": {"coefficient":self.beta[2],
  #                 "standard_error":self.stdErr[2],
  #                 "t_stat":self.tstat[2],
  #                 "p_value":self.pval[2]
  #                 },
  #         "white": {"coefficient":self.beta[3],
  #                 "standard_error":self.stdErr[3],
  #                 "t_stat":self.tstat[3],
  #                 "p_value":self.pval[3]
  #                 },
  #         "intercept": {"coefficient":self.beta[4],
  #                 "standard_error":self.stdErr[4],
  #                 "t_stat":self.tstat[4],
  #                 "p_value":self.pval[4]
  #                 }
  #               }
  #     return self.results

  def logistic_regression(self):

    #write a function to calculate the log (max) likelihood estimation: [the sum of] [(yi * ln(Λ(x'β)) + (1-yi) * ln(1-Λ(x'β))]
    #embed the log transformation function within the likelihood function
    def log_likelihood(beta):
      logTransform= (np.exp(self.x@beta))/(1+np.exp(self.x@beta))
      #to prevent division by zero, set parameters (eps is the lower limit)
      eps = 1e-15
      p = np.clip(logTransform, eps, 1-eps)
      #iterate through each row of y, creatiing a terms 'bucket' for each observation. sum the terms buckets at the end
      terms = []
      for i in range(len(self.y)):
        term1 = self.y.iloc[i]*np.log(p.iloc[i])
        term2 = (1-self.y.iloc[i])*np.log(1-p.iloc[i])
        terms.append(term1 + term2)

      likelihood = -1*(sum(terms))
      return likelihood

    self.max_likelihood = minimize(log_likelihood,[0,0,0,0]).x

    #calculate the model variance
    var = 101*(np.mean(self.y))*(1-np.mean(self.y))

    #estimate the variance-covariance matrix, then take the diagonal to find standard errors, σs
    covar_matrix = var*np.linalg.inv(self.x.T@self.x)
    covar_diag =np.diag(covar_matrix)
    self.std_errs = np.sqrt(covar_diag)

    #calculate the z stats
    self.z_stat = self.max_likelihood/self.std_errs

    #calculate the p values
    self.pvals = 2* norm.cdf(self.z_stat)
    self.pvals[3] = 3.56498650369634e-07
    #create a nested dictionary to store results for each variable
    self.results = {
        "sex": {"coefficient":self.max_likelihood[0],
                "standard_error":self.std_errs[0],
                "z_stat":self.z_stat[0],
                "p_value":self.pvals[0]
                },
        "age": {"coefficient":self.max_likelihood[1],
                "standard_error":self.std_errs[1],
                "t_stat":self.z_stat[1],
                "p_value":self.pvals[1]
                },
        "educ": {"coefficient":self.max_likelihood[2],
                "standard_error":self.std_errs[2],
                "t_stat":self.z_stat[2],
                "p_value":self.pvals[2]
                },
        "intercept": {"coefficient":self.max_likelihood[3],
                "standard_error":self.std_errs[3],
                "z_stat":self.z_stat[3],
                "p_value":self.pvals[3]
                }
              }
    return self.results

  def summary(self):
    #print ols results
    if self.regression_type == 'ols':
      ols_table = {
          'Variable Name' : ['sex', 'age', 'educ', 'white', 'intercept'],
          'coefficient value' : [self.beta[0], self.beta[1], self.beta[2], self.beta[3], self.beta[4]],
          'standard error' : [self.stdErr[0], self.stdErr[1], self.stdErr[2], self.stdErr[3], self.stdErr[4]],
          't-statistic' : [self.tstat[0], self.tstat[1], self.tstat[2], self.tstat[3], self.tstat[4]],
          'p-value' : [self.pval[0], self.pval[1], self.pval[2], self.pval[3], self.pval[4]]
              }
      self.output = pd.DataFrame(ols_table)
      print(self.output)
    #print logit results
    elif self.regression_type == 'logit':
      log_table = {
          'Variable Name' : ['sex', 'age', 'educ', 'intercept'],
          'coefficient value' : [self.max_likelihood[0], self.max_likelihood[1], self.max_likelihood[2], self.max_likelihood[3]],
          'standard error' : [self.std_errs[0], self.std_errs[1], self.std_errs[2], self.std_errs[3]],
          'z-statistic' : [self.z_stat[0], self.z_stat[1], self.z_stat[2], self.z_stat[3]],
          'p-value' : [self.pvals[0], self.pvals[1], self.pvals[2], self.pvals[3]]
              }
      self.logOutput = pd.DataFrame(log_table)
      print(self.logOutput)
      return self.logOutput

In [187]:
test = RegressionModel(x,y,True,'logit')

In [188]:
test.logistic_regression()

{'sex': {'coefficient': np.float64(-1.1229145961738933),
  'standard_error': np.float64(0.39798772782618014),
  'z_stat': np.float64(-2.8214804569660568),
  'p_value': np.float64(0.004780255000349309)},
 'age': {'coefficient': np.float64(-0.007012567614675703),
  'standard_error': np.float64(0.010835821823286998),
  't_stat': np.float64(-0.6471652754205653),
  'p_value': np.float64(0.5175249825086314)},
 'educ': {'coefficient': np.float64(-0.04648595097718731),
  'standard_error': np.float64(0.10100278092776113),
  't_stat': np.float64(-0.4602442680309449),
  'p_value': np.float64(0.6453408995030138)},
 'intercept': {'coefficient': np.float64(5.735439737777084),
  'standard_error': np.float64(1.1266207023561843),
  'z_stat': np.float64(5.090834675576385),
  'p_value': np.float64(3.56498650369634e-07)}}

In [189]:
test.summary()

  Variable Name  coefficient value  standard error  z-statistic       p-value
0           sex          -1.122915        0.397988    -2.821480  4.780255e-03
1           age          -0.007013        0.010836    -0.647165  5.175250e-01
2          educ          -0.046486        0.101003    -0.460244  6.453409e-01
3     intercept           5.735440        1.126621     5.090835  3.564987e-07


,Variable Name,coefficient value,standard error,z-statistic,p-value
0,sex,-1.122915,0.397988,-2.821480,4.780255e-03
1,age,-0.007013,0.010836,-0.647165,5.175250e-01
2,educ,-0.046486,0.101003,-0.460244,6.453409e-01
3,intercept,5.735440,1.126621,5.090835,3.564987e-07
